# Advanced Petrophysics Workflows

This tutorial covers advanced petrophysical interpretation workflows including:

- Working with shaly sand models
- Fluid contact detection
- Quality control and validation
- Custom alias configurations
- Permeability comparison
- Using the built-in plot() method

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import welly
from welly import Well, Project
from welly.petro import (
    PetroInterpreter,
    PetrophysicalParameters,
    MatrixParameters,
    FluidParameters,
    ClayParameters,
)
from welly import petro

print(f"welly version: {welly.__version__}")

## Shaly Sand Models

In shaly formations, Archie's equation overestimates water saturation. Several shaly sand models are available.

In [ ]:
# Create synthetic data for demonstration
depth = np.arange(2000, 2100, 0.5)
n = len(depth)

# Varying shale content
vsh = 0.1 + 0.3 * np.sin(np.linspace(0, 2*np.pi, n))
vsh = np.clip(vsh, 0, 0.5)

# Porosity decreasing with shale
phi = 0.25 - 0.15 * vsh + np.random.normal(0, 0.01, n)
phi = np.clip(phi, 0.05, 0.35)

# Resistivity (hydrocarbon bearing)
rt = 50 * (1 - vsh) + 5 * vsh + np.random.normal(0, 2, n)
rt = np.clip(rt, 2, 100)

# Parameters
rw = 0.05
rsh = 5.0

In [ ]:
# Compare saturation models
sw_archie = petro.archie(rt, phi, rw=rw, return_curve=False)
sw_simandoux = petro.simandoux(rt, phi, vsh, rw=rw, rsh=rsh, return_curve=False)
sw_indonesia = petro.indonesia(rt, phi, vsh, rw=rw, rsh=rsh, return_curve=False)
sw_fertl = petro.fertl(rt, phi, vsh, rw=rw, alpha=0.25, return_curve=False)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(12, 8), sharey=True)

# Track 1: Vshale
axes[0].fill_betweenx(depth, 0, vsh, color='gray', alpha=0.5)
axes[0].set_xlabel('Vshale')
axes[0].set_ylabel('Depth (m)')
axes[0].set_xlim(0, 1)

# Track 2: Porosity
axes[1].plot(phi, depth, 'b-')
axes[1].set_xlabel('Porosity')
axes[1].set_xlim(0, 0.4)

# Track 3: Sw comparison
axes[2].plot(sw_archie, depth, label='Archie', alpha=0.7)
axes[2].plot(sw_simandoux, depth, label='Simandoux', alpha=0.7)
axes[2].plot(sw_indonesia, depth, label='Indonesia', alpha=0.7)
axes[2].plot(sw_fertl, depth, label='Fertl', alpha=0.7)
axes[2].set_xlabel('Water Saturation')
axes[2].set_xlim(0, 1)
axes[2].legend(loc='lower right')

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

plt.suptitle('Shaly Sand Saturation Models Comparison')
plt.tight_layout()
plt.show()

## Waxman-Smits and Dual-Water Models

For more rigorous shaly sand analysis, the Waxman-Smits and Dual-Water models account for clay conductivity.

In [ ]:
# Waxman-Smits requires Qv (cation concentration)
# Qv can be estimated from CEC and porosity
clay_params = ClayParameters(cec=10)  # 10 meq/100g

# Compute Qv for each sample
qv = np.array([clay_params.compute_qv(p) for p in phi])

# Calculate Sw using Waxman-Smits
sw_ws = petro.waxman_smits(rt, phi, qv, rw=rw, temp=150, return_curve=False)

print(f"Waxman-Smits Sw range: {np.nanmin(sw_ws):.3f} - {np.nanmax(sw_ws):.3f}")

## Fluid Contact Detection

The petro module can detect fluid contacts from saturation profiles.

In [ ]:
# Create synthetic transition zone data
depth_tz = np.arange(2000, 2100, 0.5)

# Sw increases with depth (transition zone)
sw_tz = 0.15 + 0.01 * (depth_tz - 2000)
sw_tz = np.clip(sw_tz, 0, 1)

# Detect OWC
owc = petro.detect_owc(sw_tz, depth_tz, sw_threshold=0.5, method='threshold')
print(f"Detected OWC: {owc:.1f} m")

# Plot
fig, ax = plt.subplots(figsize=(6, 8))
ax.plot(sw_tz, depth_tz, 'b-', lw=2)
ax.axhline(owc, color='r', linestyle='--', label=f'OWC = {owc:.1f} m')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='Sw cutoff = 0.5')
ax.set_xlabel('Water Saturation')
ax.set_ylabel('Depth (m)')
ax.set_title('OWC Detection from Sw Profile')
ax.legend()
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Quality Control

The PetroInterpreter provides QC functionality to check input data quality.

In [ ]:
# Load a real well
well = Well.from_las('data/P-129_out.LAS')
interp = well.petro()

# Run QC on inputs
qc_results = interp.qc_inputs()

print("Input QC Results:")
print("=" * 50)
for curve_name, qc in qc_results.items():
    status = qc.get('status', 'unknown')
    if status == 'missing':
        print(f"{curve_name}: MISSING")
    else:
        print(f"{curve_name}: {status}")
        if 'min' in qc:
            print(f"  Range: {qc['min']:.2f} - {qc['max']:.2f}")
            print(f"  Mean: {qc['mean']:.2f}, Std: {qc['std']:.2f}")
            print(f"  NaN: {qc['nan_percent']:.1f}%")
        if qc.get('warnings'):
            print(f"  Warnings: {qc['warnings']}")

## Complete Interpretation with Built-in Plotting

Run a full interpretation and use the built-in `plot()` method for visualization.

In [ ]:
# Set up parameters
params = PetrophysicalParameters(
    matrix=MatrixParameters.sandstone(),
    fluid=FluidParameters(rw=0.05, rw_temp=75),
    clay=ClayParameters(
        gr_clean=25,
        gr_shale=130,
        nphi_shale=0.35,
        rho_shale=2.55,
        rt_shale=5.0
    ),
    a=0.81,
    m=2.0,
    n=2.0,
    name='Kennetcook Sandstone'
)

# Create interpreter and run interpretation
interp = well.petro(params=params)
results = interp.run_standard_interpretation(
    vshale_method='larionov',
    porosity_method='density',
    sw_method='archie',
    phi_cutoff=0.08,
    sw_cutoff=0.50,
    vsh_cutoff=0.40
)

print("Curves computed:", results['curves_computed'])

In [ ]:
# Use the built-in plot() method
fig = interp.plot()
plt.show()

In [ ]:
# Plot a specific depth interval
fig = interp.plot(depth_range=(1500, 1800), title='Reservoir Interval')
plt.show()

In [ ]:
# Plot only computed results (no input curves)
fig = interp.plot(show_inputs=False, title='Interpretation Results')
plt.show()

## Buckles Analysis

Buckles number (BVW = φ × Sw) is useful for identifying transition zones and irreducible water saturation.

In [ ]:
# Calculate Buckles number using the interpreter
bvw = interp.buckles()

# In transition zone, BVW should be approximately constant
print(f"BVW range: {np.nanmin(bvw.values):.4f} - {np.nanmax(bvw.values):.4f}")
print(f"BVW mean: {np.nanmean(bvw.values):.4f}")
print(f"BVW std: {np.nanstd(bvw.values):.4f}")

## Custom Alias Configuration

Different datasets use different curve mnemonics. The alias system handles this.

In [ ]:
# Create a custom alias configuration
custom_aliases = {
    'GR': ['GR', 'GRGC', 'GRD', 'GAMMA', 'GAMMA_RAY'],
    'RHOB': ['RHOB', 'DEN', 'DENS', 'DENSITY', 'RHOZ'],
    'NPHI': ['NPHI', 'NEU', 'NPOR', 'TNPH', 'NEUTRON'],
    'RT': ['RT', 'ILD', 'RILD', 'RD', 'RLLD', 'LLD', 'RESD'],
    'DT': ['DT', 'DTC', 'AC', 'DTCO', 'SONIC'],
}

# Create interpreter with custom aliases
interp_custom = PetroInterpreter(well, alias=custom_aliases)

# Check matches
matches = interp_custom.get_alias_matches()
print("Alias matches with custom configuration:")
for std, actual in matches.items():
    print(f"  {std}: {actual}")

In [ ]:
# Save aliases to file for reuse
interp_custom.save_aliases('data/my_aliases.json', description='Custom aliases for my dataset')
print("Aliases saved to data/my_aliases.json")

# Load in another interpreter
interp2 = PetroInterpreter(well)
interp2.load_aliases('data/my_aliases.json')
print("Aliases loaded successfully")

## Permeability Comparison

Different permeability correlations can give very different results. It's important to calibrate with core data.

In [ ]:
# Compare permeability methods using synthetic data
k_timur = petro.perm_timur(phi, sw_archie, return_curve=False)
k_coates = petro.perm_coates(phi, sw_archie, return_curve=False)
k_tixier = petro.perm_tixier(phi, sw_archie, return_curve=False)

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(k_timur, depth, label='Timur', alpha=0.7)
ax.plot(k_coates, depth, label='Coates', alpha=0.7)
ax.plot(k_tixier, depth, label='Tixier', alpha=0.7)

ax.set_xlabel('Permeability (mD)')
ax.set_ylabel('Depth (m)')
ax.set_xscale('log')
ax.set_title('Permeability Correlation Comparison')
ax.legend()
ax.invert_yaxis()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Saving Interpretation Plots

The `plot()` method returns a matplotlib Figure that can be saved.

In [ ]:
# Create a publication-quality figure
fig = interp.plot(
    figsize=(18, 14),
    title=f'Complete Petrophysical Interpretation: {well.name}'
)
fig.savefig('data/advanced_interpretation.png', dpi=150, bbox_inches='tight')
print("Saved: data/advanced_interpretation.png")
plt.show()

## Summary

This tutorial covered:

- **Shaly sand models**: Simandoux, Indonesia, Fertl, Waxman-Smits for formations with clay
- **Fluid contacts**: Detecting OWC, GOC, FWL from saturation profiles
- **Quality control**: Checking input data quality before interpretation
- **Built-in plotting**: Using `interp.plot()` for standard interpretation displays
- **Buckles analysis**: Using BVW to identify transition zones
- **Custom aliases**: Configuring curve name mappings for different datasets
- **Permeability comparison**: Understanding differences between correlations